In [59]:
import torch.nn as nn
import pandas as pd
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torch._dynamo
torch._dynamo.config.suppress_errors = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

In [60]:
train_df = pd.read_csv("data/asl_data/sign_mnist_train.csv")
valid_df = pd.read_csv("data/asl_data/sign_mnist_valid.csv")

In [61]:
sample_df = train_df.head().copy()  # Grab the top 5 rows
sample_df.pop('label')
sample_x = sample_df.values
sample_x

array([[107, 118, 127, ..., 204, 203, 202],
       [155, 157, 156, ..., 103, 135, 149],
       [187, 188, 188, ..., 195, 194, 195],
       [211, 211, 212, ..., 222, 229, 163],
       [164, 167, 170, ..., 163, 164, 179]], shape=(5, 784))

In [62]:
sample_x.shape

(5, 784)

In [63]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1

sample_x = sample_x.reshape(-1, IMG_CHS, IMG_HEIGHT, IMG_WIDTH)
sample_x.shape

(5, 1, 28, 28)

In [64]:

class MyDataset(Dataset):
    def __init__(self, base_df):
        x_df = base_df.copy()  # Some operations below are in-place
        y_df = x_df.pop('label')
        x_df = x_df.values / 255  # Normalize values from 0 to 1
        x_df = x_df.reshape(-1, IMG_CHS, IMG_WIDTH, IMG_HEIGHT)
        self.xs = torch.tensor(x_df).float().to(device)
        self.ys = torch.tensor(y_df).to(device)

    def __getitem__(self, idx):
        x = self.xs[idx]
        y = self.ys[idx]
        return x, y

    def __len__(self):
        return len(self.xs)

In [65]:
BATCH_SIZE = 32

train_data = MyDataset(train_df)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
train_N = len(train_loader.dataset)

valid_data = MyDataset(valid_df)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)
valid_N = len(valid_loader.dataset)

In [66]:
batch = next(iter(train_loader))
batch

[tensor([[[[0.4392, 0.4549, 0.4902,  ..., 0.6118, 0.6000, 0.6000],
           [0.4510, 0.4745, 0.5137,  ..., 0.6196, 0.6078, 0.6039],
           [0.4706, 0.5020, 0.5412,  ..., 0.6353, 0.6275, 0.6196],
           ...,
           [0.7137, 0.7569, 0.7843,  ..., 0.0000, 0.0667, 0.0784],
           [0.7216, 0.7647, 0.8000,  ..., 0.0235, 0.0118, 0.0588],
           [0.5608, 0.5922, 0.6118,  ..., 0.0510, 0.0314, 0.0196]]],
 
 
         [[[0.5490, 0.5608, 0.5765,  ..., 0.5961, 0.5961, 0.5922],
           [0.5569, 0.5686, 0.5804,  ..., 0.6039, 0.6000, 0.5961],
           [0.5608, 0.5765, 0.5961,  ..., 0.6196, 0.6157, 0.6078],
           ...,
           [0.6706, 0.6824, 0.7020,  ..., 0.7647, 0.7569, 0.7529],
           [0.6706, 0.6863, 0.7020,  ..., 0.7686, 0.7608, 0.7569],
           [0.6706, 0.6863, 0.7020,  ..., 0.7686, 0.7608, 0.7569]]],
 
 
         [[[0.5686, 0.5804, 0.6000,  ..., 0.6902, 0.6863, 0.6824],
           [0.5686, 0.5843, 0.6000,  ..., 0.6941, 0.6902, 0.6863],
           [0.5725

In [67]:
batch[0].shape

torch.Size([32, 1, 28, 28])

In [68]:
batch[1].shape

torch.Size([32])

In [69]:
n_classes = 24
kernel_size = 3
flattened_img_size = 75 * 3 * 3

model = nn.Sequential(
    # First convolution
    nn.Conv2d(IMG_CHS, 25, kernel_size, stride=1, padding=1),  # 25 x 28 x 28
    nn.BatchNorm2d(25),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 25 x 14 x 14
    # Second convolution
    nn.Conv2d(25, 50, kernel_size, stride=1, padding=1),  # 50 x 14 x 14
    nn.BatchNorm2d(50),
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2, stride=2),  # 50 x 7 x 7
    # Third convolution
    nn.Conv2d(50, 75, kernel_size, stride=1, padding=1),  # 75 x 7 x 7
    nn.BatchNorm2d(75),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 75 x 3 x 3
    # Flatten to Dense
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512, n_classes)
)

In [70]:
try:
    model = torch.compile(model.to(device))
except:
    model = model.to(device)

In [71]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters())

In [72]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

In [73]:

def validate():
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [74]:

def train():
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [75]:
epochs = 20

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()

Epoch: 0
Train - Loss: 284.1780 Accuracy: 0.9016
Valid - Loss: 35.2460 Accuracy: 0.9525
Epoch: 1
Train - Loss: 17.9558 Accuracy: 0.9948
Valid - Loss: 19.2345 Accuracy: 0.9677
Epoch: 2
Train - Loss: 13.8220 Accuracy: 0.9951
Valid - Loss: 22.9033 Accuracy: 0.9582
Epoch: 3
Train - Loss: 8.5835 Accuracy: 0.9971
Valid - Loss: 20.8443 Accuracy: 0.9672
Epoch: 4
Train - Loss: 10.9513 Accuracy: 0.9961
Valid - Loss: 19.5406 Accuracy: 0.9727
Epoch: 5
Train - Loss: 0.8977 Accuracy: 0.9998
Valid - Loss: 17.5172 Accuracy: 0.9736
Epoch: 6
Train - Loss: 0.1808 Accuracy: 1.0000
Valid - Loss: 20.7307 Accuracy: 0.9716
Epoch: 7
Train - Loss: 0.0737 Accuracy: 1.0000
Valid - Loss: 15.0511 Accuracy: 0.9757
Epoch: 8
Train - Loss: 20.2035 Accuracy: 0.9935
Valid - Loss: 25.6125 Accuracy: 0.9618
Epoch: 9
Train - Loss: 7.0108 Accuracy: 0.9972
Valid - Loss: 40.0450 Accuracy: 0.9498
Epoch: 10
Train - Loss: 3.1390 Accuracy: 0.9990
Valid - Loss: 14.1076 Accuracy: 0.9757
Epoch: 11
Train - Loss: 7.6412 Accuracy: 0.9972